# 多重共線性・正則化・モデル選択（最小構成版）

シンプルな合成データ1つで、多重共線性・Ridge(L2)・Lasso(L1)・AIC/BIC を体験します。
各セクションは5〜10行のコードで完結します。


## 0. 準備

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import Ridge, Lasso

np.random.seed(0)
plt.rcParams['axes.unicode_minus'] = False


## 1. データ生成

`x1` と `x2` をほぼ同じ情報（相関 0.95）にし、`x3` は無関係にする。  
真のモデル：$y = 3 + 2x_1 - 1 \cdot x_3 + \varepsilon$（**x2 は本来 y に無関係**）。


In [ ]:
n = 100
x1 = np.random.normal(50, 10, n)
x2 = 0.95 * x1 + np.random.normal(0, 3, n)   # x1とほぼ同じ情報（多重共線性の原因）
x3 = np.random.normal(20, 5, n)               # 独立な変数
y  = 3 + 2*x1 - 1*x3 + np.random.normal(0, 5, n)

df = pd.DataFrame({'y': y, 'x1': x1, 'x2': x2, 'x3': x3})
df.head()


## 2. 相関を見る

`x1` と `x2` の相関係数を確認する。


In [ ]:
print(df.corr().round(2))


## 3. 重回帰（OLS）― 全変数投入

`x2` は本来 `y` に無関係なのに、モデルに入れるとどうなるか。


In [ ]:
X = sm.add_constant(df[['x1', 'x2', 'x3']])
ols = sm.OLS(df['y'], X).fit()
print(ols.summary().tables[1])


**観察：** `x1` の係数（真値=2）が不安定になり、`x2` の p値が非有意になっていないか確認しよう。  
`x1` と `x2` のどちらが効いているのか、モデルが「決められない」状態になっている。


## 4. VIF（多重共線性の数値診断）

$$\text{VIF}_j = \frac{1}{1-R_j^2}$$

**VIF > 10** で多重共線性が疑われる。


In [ ]:
vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    index=X.columns, name='VIF'
)
print(vif.round(1))


## 5. 正則化（Ridge=L2 / Lasso=L1）

正則化はペナルティで係数を縮小し、多重共線性による不安定さを抑える。  
**Lasso は不要な変数の係数をゼロにする**（自動的な変数選択）。


In [ ]:
X_vars = df[['x1', 'x2', 'x3']].values
y_vals = df['y'].values

ridge = Ridge(alpha=10).fit(X_vars, y_vals)
lasso = Lasso(alpha=1.0).fit(X_vars, y_vals)

coef_df = pd.DataFrame({
    'OLS':   ols.params[1:].values,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_,
}, index=['x1', 'x2', 'x3'])
print(coef_df.round(3))


In [ ]:
coef_df.plot.bar(figsize=(6, 3.5), color=['#2980b9', '#e67e22', '#27ae60'])
plt.axhline(0, color='black', lw=0.8)
plt.ylabel('係数')
plt.title('OLS / Ridge / Lasso の係数比較')
plt.tight_layout()
plt.show()


**観察：** Lasso は `x2`（無関係な変数）の係数をゼロに近づけている。  
Ridge は係数を縮小するが、ゼロにはしない。


## 6. AIC / BIC によるモデル選択

「全変数モデル」と「x2を除いたモデル」を比較する。


In [ ]:
m_full = sm.OLS(df['y'], sm.add_constant(df[['x1','x2','x3']])).fit()
m_drop = sm.OLS(df['y'], sm.add_constant(df[['x1','x3']])).fit()

result = pd.DataFrame({
    'モデル':  ['全変数 (x1,x2,x3)', 'x2を除く (x1,x3)'],
    'AIC':     [m_full.aic, m_drop.aic],
    'BIC':     [m_full.bic, m_drop.bic],
}).round(1)
print(result)


**結論：** AIC・BIC は両方とも「x2を除いたモデル」を選ぶはず（値が小さい方が良い）。  
これは Lasso が x2 をゼロにした判断と一致する。

---

## まとめ

| 概念 | 確認したこと |
|------|------------|
| 多重共線性 | x1・x2 の高相関で OLS の係数が不安定化 |
| VIF | 数値で多重共線性を検出（VIF > 10） |
| Ridge (L2) | 係数を縮小して安定化（ゼロにはしない） |
| Lasso (L1) | 不要変数の係数をゼロにして自動選択 |
| AIC/BIC | モデル比較で「不要な変数を含むモデル」を排除 |

### 演習
1. `x1`・`x2` の相関を `0.95` → `0.99` に変えると VIF はどう変わるか？
2. Ridge の `alpha` を 0.1, 1, 10, 100 と変えて係数の変化を見てみよう。
